### using SGD as the optimizer for Fashion-MNIST Classification

In [56]:
# implemented by HuyIGW04
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms as tf    # for convert tensor
from torch.utils.data import DataLoader

In [57]:
# setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [58]:
train_set = datasets.FashionMNIST(root='FashionMNIST-data',
                                  train=True,
                                  transform=tf.ToTensor(),
                                  download='True')

test_set = datasets.FashionMNIST(root='FashionMNIST-data',
                                 train=False,
                                 transform=tf.ToTensor(),
                                 download=True)

print(train_set.classes)

['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [59]:
# MNIST data, ~ 1024
train_dataloader = DataLoader(train_set, 
                              batch_size=1024,
                              shuffle=True)

test_dataloader = DataLoader(test_set,
                             batch_size=1024)

train_dataloader.dataset

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: FashionMNIST-data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [60]:
# basic custom: 784:128:64:10
class customNN(nn.Module):
    def __init__(self):
        super(customNN, self).__init__()
        
        self.flatten = nn.Flatten()
        self.MLP = nn.Sequential(
            nn.Linear(784, 128),         # 28 x 28
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)           # 10 classes
        )
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.MLP(x)
        return x

model = customNN().to(device)
print(model)

customNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (MLP): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [61]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [62]:
def training(dataloader, model, loss_fn, optimizer):
    """training field"""
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # forward
        X, y = X.to(device), y.to(device)
        y_hat = model(X)
        loss = loss_fn(y_hat, y)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # for debug
        if batch % 100 == 0:
            loss_print = loss.item()
            print(f'loss: {loss_print}')



def testing_and_metric(dataloader_obj, model, loss_fn):
    """calculate test loss and accuracy metric"""
    num_of_batch = len(dataloader_obj)
    num_of_pic = len(dataloader_obj.dataset)
    loss_test_sum = 0                                                            # calculate mean    
    acc_sum = 0                                                                  # calculate accuracy

    with torch.no_grad():
        for batch, (X, y) in enumerate(dataloader_obj):
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss_test_sum += loss.item()                                         # for loss value
            
            acc_sum += (y == pred.argmax(1)).type(torch.float).sum().item()      # class -> dim=1

    model.eval()
    loss_test_mean = loss_test_sum/num_of_batch
    acc_mean = acc_sum/num_of_pic
    print(f"Test Error:\n Accuracy: {100*acc_mean:>.2f}%,      Loss: {loss_test_mean:>.5f} \n")




In [63]:
epoch = 50
for i in range(epoch):
    print(f"Epoch {i+1}\n---------------------")
    training(train_dataloader, model, criterion, optimizer)
    testing_and_metric(test_dataloader, model, criterion)
print('Done!')

Epoch 1
---------------------


loss: 2.3038129806518555
Test Error:
 Accuracy: 18.41%,      Loss: 2.25936 

Epoch 2
---------------------
loss: 2.2551167011260986
Test Error:
 Accuracy: 21.42%,      Loss: 2.19456 

Epoch 3
---------------------
loss: 2.1962404251098633
Test Error:
 Accuracy: 30.14%,      Loss: 2.09107 

Epoch 4
---------------------
loss: 2.100191831588745
Test Error:
 Accuracy: 39.53%,      Loss: 1.93696 

Epoch 5
---------------------
loss: 1.9412753582000732
Test Error:
 Accuracy: 44.83%,      Loss: 1.74041 

Epoch 6
---------------------
loss: 1.7499959468841553
Test Error:
 Accuracy: 52.95%,      Loss: 1.54361 

Epoch 7
---------------------
loss: 1.5504794120788574
Test Error:
 Accuracy: 57.14%,      Loss: 1.38055 

Epoch 8
---------------------
loss: 1.3885043859481812
Test Error:
 Accuracy: 60.58%,      Loss: 1.25470 

Epoch 9
---------------------
loss: 1.2455875873565674
Test Error:
 Accuracy: 62.73%,      Loss: 1.15745 

Epoch 10
---------------------
loss: 1.1600072383880615
Test Error:
